# Mobile Price predictor

## Initialise

In [1]:
!pip install kagglehub
!pip install sweetviz

In [2]:
import pandas as pd
import numpy as np
import kagglehub
from pathlib import Path
import os,re 

In [3]:
path = kagglehub.dataset_download("abhishekgupta56447/flipkart-mobile-phones-dataset")
path = Path(path)
os.listdir(path)

['Flipkart Phone Data.csv']

In [4]:
df = pd.read_csv(path / 'Flipkart Phone Data.csv')

In [5]:
df

,Product_Name,Rating,Number_of_Ratings,Number_of_Review,Display_Details,Camera_Details,Battery_Details,Processor_Details,Warranty_Details,Selling_Price,Original_Price,Discount,RAM,ROM,Expandable_Storage
0,"Acer Super ZX 5G (Lunar Blue, 128 GB)",4.1,172,13,17.02 cm (6.7 inch) Display,64MP Rear Camera,5000 mAh Battery,NaN,1 Year Manufacturer,"â‚¹14,499","â‚¹23,999",39% off,4 GB RAM,128 GB ROM,NaN
1,"Ai+ Nova 2 5G (Black, 128 GB)",4.2,"3,818",235,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,T8200 Processor,1 Year Warranty on Handset and 6 Months Warran...,"â‚¹13,999","â‚¹14,999",6% off,6 GB RAM,128 GB ROM,Expandable Upto 1 TB
2,"Ai+ Nova 2 5G (Black, 64 GB)",4.4,"4,095",167,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,T8200 Processor,1 Year Warranty on Handset and 6 Months Warran...,"â‚¹11,999","â‚¹13,999",14% off,4 GB RAM,64 GB ROM,Expandable Upto 1 TB
3,"Ai+ Nova 2 5G (Blue, 128 GB)",4.4,"4,095",167,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,T8200 Processor,1 Year Warranty on Handset and 6 Months Warran...,"â‚¹12,999","â‚¹17,499",25% off,4 GB RAM,128 GB ROM,Expandable Upto 1 TB
4,"Ai+ Nova 2 5G (Blue, 128 GB)",4.2,"3,818",235,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,T8200 Processor,1 Year Warranty on Handset and 6 Months Warran...,"â‚¹13,999","â‚¹14,999",6% off,6 GB RAM,128 GB ROM,Expandable Upto 1 TB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,"vivo Y21 5G (Midnight Blue, 128 GB)",4.6,81,5,17.12 cm (6.74 inch) Full HD Display,50MP + 0.08MP | 5MP Front Camera,6500 mAh Li-ion Battery,Dimensity 6300 Processor,1 Year Manufacturer Warranty for Device and 6 ...,"â‚¹23,999","â‚¹35,999",33% off,6 GB RAM,128 GB ROM,Expandable Upto 2 TB
786,"vivo Y21 5G (Mystic Blue, 128 GB)",4.1,98,7,17.12 cm (6.74 inch) Full HD Display,50MP + 0.08MP | 5MP Front Camera,6500 mAh Li-ion Battery,Dimensity 6300 Processor,1 Year Manufacturer Warranty for Device and 6 ...,"â‚¹20,999","â‚¹49,999",58% off,4 GB RAM,128 GB ROM,Expandable Upto 2 TB
787,"vivo Y21 5G (Mystic Blue, 128 GB)",4.6,81,5,17.12 cm (6.74 inch) Full HD Display,50MP + 0.08MP | 5MP Front Camera,6500 mAh Li-ion Battery,Dimensity 6300 Processor,1 Year Manufacturer Warranty for Device and 6 ...,"â‚¹23,999","â‚¹54,999",56% off,6 GB RAM,128 GB ROM,Expandable Upto 2 TB
788,"vivo Y21 5G (Straw Gold, 128 GB)",4.1,98,7,17.12 cm (6.74 inch) Full HD Display,50MP + 0.08MP | 5MP Front Camera,6500 mAh Li-ion Battery,Dimensity 6300 Processor,1 Year Manufacturer Warranty for Device and 6 ...,"â‚¹20,999","â‚¹49,999",58% off,4 GB RAM,128 GB ROM,Expandable Upto 2 TB


In [6]:
print(df.shape)

(790, 15)


In [7]:
df.dtypes

Product_Name              str
Rating                float64
Number_of_Ratings         str
Number_of_Review          str
Display_Details           str
Camera_Details            str
Battery_Details           str
Processor_Details         str
Warranty_Details          str
Selling_Price             str
Original_Price            str
Discount                  str
RAM                       str
ROM                       str
Expandable_Storage        str
dtype: object

In [8]:
df.duplicated().sum()

np.int64(0)

## Data Cleaning & Integrity

In [9]:
df1 = df


In [10]:

def convert_to_MB(value):
    if isinstance(value, str):
        v  = float(re.search(r"\d+", value).group())
        if "GB" in value:
            return float(v) * 1024
        elif "MB" in value:
            return float(v)
    return np.nan

In [11]:
### Cleaning Display details
d = df1['Display_Details'].str.extract("^(?P<size>\d+.?(\d+)?)\s*cm\s*\((?P<size_inch>\d+.?(\d+)?)\s*inch\)\s*(?P<tech>.*?)\sDisplay$")
d = d[['size', "tech"]].rename(columns={"size": "display_size", "tech": "display_tech"})
df = df1.join(d)

### Camera_Details
d = df["Camera_Details"].str.split("|", expand=True)
d.rename(columns={1:"front_camera"}, inplace=True)
front = d[["front_camera"]]
rear = d[0].str.split("+",expand=True)
for x in rear.columns:
    rear[x] = rear[x].str.extract("(\d+\.?\d*)").astype(float)


rear.columns = "rear_" + rear.columns.astype(str)
df["front_camera"] = front["front_camera"].str.extract("(\d+\.?\d*)").astype(float)
df = df.join(rear)

## Battery Details
df["Battery_Details"] = df["Battery_Details"].str.extract("(\d+\.?\d*)")

## Warranty_Details
d = df["Warranty_Details"].str.replace(r"\W|\s", "", regex=True).str.lower()
d = pd.DataFrame(d.unique())
number_map = {"one1": "1","one": "1", "two":"2", "three":"3", "four":"4", "five":"5", "six":"6", "seven": "7", "eight":"8" ,"nine":"9"}
for x,y in number_map.items():
    d[0] = d[0].str.replace(x,y)
d = d[0].str.extract("(?P<warantyperiod>\d+)\s*(?P<mult>(?:years?|yrs?|months?|mon(?:th)?s?))", expand=True)
mult_map = {"year": 12, "years": 12, "yr": 12, "yrs": 12, "month": 1, "months": 1, "mon": 1, "mons": 1}
d["mult"] = d["mult"].map(mult_map)
d["warantyperiod"] = d["warantyperiod"].astype(float) * d["mult"]
df = df.join(d[["warantyperiod"]])  

## Selling_Price
df["Selling_Price"] = df["Selling_Price"].str.replace(r"\D", "", regex=True).astype(float)

## Original_Price
df["Original_Price"] = df["Original_Price"].str.replace(r"\D", "", regex=True).astype(float)


## Discount
df["Discount"] = df["Discount"].str.replace(r"\D", "", regex=True).astype(float)

##RAM AND ROM
df["RAM"] = df["RAM"].apply(convert_to_MB)
df["ROM"] = df["ROM"].apply(convert_to_MB)
df["Expandable_Storage"] = df["Expandable_Storage"].apply(convert_to_MB)

df.drop(columns=['Display_Details' , "Camera_Details", "Warranty_Details"], inplace=True)


In [12]:
df

,Product_Name,Rating,Number_of_Ratings,Number_of_Review,Battery_Details,Processor_Details,Selling_Price,Original_Price,Discount,RAM,ROM,Expandable_Storage,display_size,display_tech,front_camera,rear_0,rear_1,rear_2,rear_3,warantyperiod
0,"Acer Super ZX 5G (Lunar Blue, 128 GB)",4.1,172,13,5000,NaN,14499.0,23999.0,39.0,4096.0,131072.0,NaN,17.02,,NaN,64.0,NaN,NaN,NaN,12.0
1,"Ai+ Nova 2 5G (Black, 128 GB)",4.2,"3,818",235,6000,T8200 Processor,13999.0,14999.0,6.0,6144.0,131072.0,NaN,17.13,HD+,8.0,50.0,NaN,NaN,NaN,12.0
2,"Ai+ Nova 2 5G (Black, 64 GB)",4.4,"4,095",167,6000,T8200 Processor,11999.0,13999.0,14.0,4096.0,65536.0,NaN,17.13,HD+,8.0,50.0,NaN,NaN,NaN,12.0
3,"Ai+ Nova 2 5G (Blue, 128 GB)",4.4,"4,095",167,6000,T8200 Processor,12999.0,17499.0,25.0,4096.0,131072.0,NaN,17.13,HD+,8.0,50.0,NaN,NaN,NaN,NaN
4,"Ai+ Nova 2 5G (Blue, 128 GB)",4.2,"3,818",235,6000,T8200 Processor,13999.0,14999.0,6.0,6144.0,131072.0,NaN,17.13,HD+,8.0,50.0,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,"vivo Y21 5G (Midnight Blue, 128 GB)",4.6,81,5,6500,Dimensity 6300 Processor,23999.0,35999.0,33.0,6144.0,131072.0,NaN,17.12,Full HD,5.0,50.0,0.08,NaN,NaN,NaN
786,"vivo Y21 5G (Mystic Blue, 128 GB)",4.1,98,7,6500,Dimensity 6300 Processor,20999.0,49999.0,58.0,4096.0,131072.0,NaN,17.12,Full HD,5.0,50.0,0.08,NaN,NaN,NaN
787,"vivo Y21 5G (Mystic Blue, 128 GB)",4.6,81,5,6500,Dimensity 6300 Processor,23999.0,54999.0,56.0,6144.0,131072.0,NaN,17.12,Full HD,5.0,50.0,0.08,NaN,NaN,NaN
788,"vivo Y21 5G (Straw Gold, 128 GB)",4.1,98,7,6500,Dimensity 6300 Processor,20999.0,49999.0,58.0,4096.0,131072.0,NaN,17.12,Full HD,5.0,50.0,0.08,NaN,NaN,NaN
